# Qiskit LEAN Verify Game

Circuit puzzles graded by Qiskit's own `Statevector` simulation, with the
underlying gate identities independently proven in
[CliffordGame](https://github.com/RexRowan/CliffordGame) — a companion
[lean4game](https://github.com/leanprover-community/lean4game) submission
where Lean's kernel is the referee instead of Qiskit.

**What "cross-checked" means here:** this package's Qiskit result and
CliffordGame's Lean-proven result are shown to agree for each of the 7
levels below — not a claim that Qiskit's simulator itself is formally
verified. See [`docs/LIMITATIONS.md`](https://github.com/RexRowan/Qiskit-LEAN-Verify-Game/blob/main/docs/LIMITATIONS.md)
for the precise scope.

Run this notebook top to bottom. The gate-button widgets in the later
cells need a live kernel (they won't render in a static GitHub preview) —
open this in Colab or Jupyter to actually play the levels.


## 1. Install

In [ ]:
%%capture
!pip install "qiskit>=2.0" ipywidgets
!pip install "qiskit_lean_verify_game[widget] @ git+https://github.com/RexRowan/Qiskit-LEAN-Verify-Game.git"


In [ ]:
import qiskit
import qiskit_lean_verify_game

print("Qiskit version:", qiskit.__version__)
print("qiskit_lean_verify_game version:", qiskit_lean_verify_game.__version__)


## 2. The levels

Seven levels, all in the Clifford + S fragment — no continuous-parameter
gates, so every check here is decidable rather than approximate.


In [ ]:
from qiskit_lean_verify_game.levels import LEVELS

for level in LEVELS:
    print(f"{level.id:28s} {level.title}")


## 3. Grade a circuit programmatically (no widget)

The same check the widget UI runs under the hood: build a circuit from a
level's allowed gates, then grade it against the level's target.


In [ ]:
from qiskit_lean_verify_game.levels import get_level, build_circuit
from qiskit_lean_verify_game.grader import grade_circuit

level = get_level("level07_bell_state")
print(level.title, "-", level.goal)

circuit = build_circuit(level, [("H", 0), ("CNOT", 0, 1)])
print(circuit.draw(output="text"))

result = grade_circuit(circuit, level.target, initial_state=level.initial_state,
                        phase_sensitive=level.phase_sensitive)
print("PASSED" if result.passed else "NOT YET", "-", result.reason)


## 4. Play a single level interactively

Click the gate buttons to build a circuit, then **Check**.


In [ ]:
from qiskit_lean_verify_game.widget import LevelWidget
from qiskit_lean_verify_game.levels import get_level

LevelWidget(get_level("level07_bell_state")).show()


## 5. Play through all seven levels

`GameShell` adds a level-select row on top of `LevelWidget` — click a
level to switch to it, solved levels get a checkmark.


In [ ]:
from qiskit_lean_verify_game.widget import GameShell

GameShell().show()


## 6. Cross-check against Lean (optional, needs a local Lean toolchain)

This step doesn't run in Colab by default — it shells out to a Lean 4
compiler to independently re-derive each level's result and diff it
against Qiskit's. Skip this cell unless you have Lean installed; it's
here for completeness, not part of the interactive demo.

```bash
# Requires a Lean 4 toolchain matching scripts/lean_reference/lean-toolchain
!python scripts/crosscheck_against_lean.py
```


## See also

- [CliffordGame](https://github.com/RexRowan/CliffordGame) — the companion Lean proof, playable as a [lean4game](https://github.com/leanprover-community/lean4game) submission
- [qiskit-zx-verified](https://github.com/RexRowan/qiskit-zx-verified) — the earlier Lean/Python cross-check this package's discipline follows
